# Model predykcyjny oparty na regresji liniowej

Celem analizy jest stworzenie modelu umożliwiającego predykcje ceny samochodu w zależności od jego specyfikacji. Lista potemcjalnych regresorów:
* Marka -> bazując na wiedzy rynku samochodowego niektóre marki do co zasady są droższe od innych np. BMW jest droższe od Renault
* Przebieg -> bazując na intuicji, im niższy przebieg, tym droższy samochód powinien być
* Moc silnika -> droższe samochody powinny mieć mocniejszy silnik
* Rocznik -> poza outlierami im starsze auto tym tańsze powinno być

## Przygotowanie danych

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
sns.set_theme(style="whitegrid")

In [13]:
data = pd.read_csv('cars_data.csv')
data

,Marka,Model,Cena,Waluta,Paliwo,Przebieg,Rocznik,Skrzynia biegów,Pojemność silnika
0,Alfa Romeo,Alfa Romeo 147,3000,PLN,Benzyna,159000,2008,Manualna,NaN
1,Volvo,Volvo XC 60,129999,PLN,Benzyna,127000,2018,Automatyczna,NaN
2,Ford,Ford B-MAX,22900,PLN,Benzyna,161000,2016,Manualna,NaN
3,Jaguar,Jaguar I-Pace EV400 AWD SE,69900,PLN,Elektryczny,135125,2019,Automatyczna,NaN
4,Cupra,Cupra Formentor VZ 2.0 TSI 4Drive DSG,119999,PLN,Benzyna,81800,2020,Automatyczna,2.0
...,...,...,...,...,...,...,...,...,...
9652,SsangYong/KGM,SsangYong/KGM Tivoli,35550,PLN,Benzyna,76000,2015,Manualna,NaN
9653,Mercedes-Benz,Mercedes-Benz GLE,280000,PLN,Benzyna,61000,2022,Automatyczna,NaN
9654,Land Rover,Land Rover Range Rover,79900,PLN,Diesel,290000,2014,Automatyczna,NaN
9655,Volkswagen,Volkswagen T-Roc 1.5 TSI R-Line DSG,109900,PLN,Benzyna,14200,2024,Automatyczna,1.5


In [14]:
data.describe(include='all')

,Marka,Model,Cena,Waluta,Paliwo,Przebieg,Rocznik,Skrzynia biegów,Pojemność silnika
count,9657,9657,9.657000e+03,9657,9657,9.657000e+03,9657.000000,9657,4139.000000
unique,94,5630,NaN,1,7,NaN,NaN,2,NaN
top,BMW,Opel Astra,NaN,PLN,Benzyna,NaN,NaN,Manualna,NaN
freq,725,58,NaN,9657,5201,NaN,NaN,4838,NaN
mean,NaN,NaN,7.703648e+04,NaN,NaN,1.422563e+05,2016.261158,NaN,1.669896
std,NaN,NaN,1.000799e+05,NaN,NaN,9.776860e+04,7.001371,NaN,0.517889
min,NaN,NaN,1.350000e+03,NaN,NaN,0.000000e+00,1952.000000,NaN,1.000000
25%,NaN,NaN,2.250000e+04,NaN,NaN,6.732700e+04,2012.000000,NaN,1.400000
50%,NaN,NaN,4.690000e+04,NaN,NaN,1.399600e+05,2017.000000,NaN,1.600000
75%,NaN,NaN,9.490000e+04,NaN,NaN,2.025000e+05,2022.000000,NaN,2.000000


In [15]:
data.isnull().sum()

Marka                   0
Model                   0
Cena                    0
Waluta                  0
Paliwo                  0
Przebieg                0
Rocznik                 0
Skrzynia biegów         0
Pojemność silnika    5518
dtype: int64

## Feature engineering zmiennej Pojemność silnika

Jako że 57% danych w tej kolumnie numerycznej stanowią braki, zastosowane zostanie dwustopniowe podejście:
1. Utworzenie kolumny pomocniczej: Pojemność znana
2. Imputacja braków za pomocą mediany grupy + zachowanie zmiennej

In [16]:
df = pd.read_csv("cars_data.csv")

# Sprawdźmy, czy dla brakujących wierszy możemy odtworzyć pojemność z innych aut tej samej marki i modelu
# (jeśli dany model ma w bazie wersje z znaną pojemnością)
print("Liczba unikalnych modeli z brakami:", df[df['Pojemność silnika'].isnull()]['Model'].nunique())

Liczba unikalnych modeli z brakami: 2381
